In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# -----------------------------
# Data Augmentation
# -----------------------------
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    transforms.RandomErasing(p=0.2)
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=False, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

# -----------------------------
# ADMM Layers
# -----------------------------
class ADMMConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, rho=1e-4):
        super(ADMMConv2d, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=True)
        self.rho = rho
        self.register_buffer("Z", torch.zeros_like(self.conv.weight))
        self.register_buffer("U", torch.zeros_like(self.conv.weight))

    def forward(self, x):
        return self.conv(x)

    def admm_loss(self):
        return (self.rho / 2) * torch.norm(self.conv.weight - self.Z + self.U) ** 2

    def update_ZU(self):
        with torch.no_grad():
            temp = self.conv.weight + self.U
            threshold = self.rho
            self.Z = torch.sign(temp) * torch.clamp(torch.abs(temp) - threshold, min=0)
            self.U = self.U + (self.conv.weight - self.Z)


class ADMMLinear(nn.Module):
    def __init__(self, in_features, out_features, rho=1e-4):
        super(ADMMLinear, self).__init__()
        self.fc = nn.Linear(in_features, out_features)
        self.rho = rho
        self.register_buffer("Z", torch.zeros_like(self.fc.weight))
        self.register_buffer("U", torch.zeros_like(self.fc.weight))

    def forward(self, x):
        return self.fc(x)

    def admm_loss(self):
        return (self.rho / 2) * torch.norm(self.fc.weight - self.Z + self.U) ** 2

    def update_ZU(self):
        with torch.no_grad():
            temp = self.fc.weight + self.U
            threshold = self.rho
            self.Z = torch.sign(temp) * torch.clamp(torch.abs(temp) - threshold, min=0)
            self.U = self.U + (self.fc.weight - self.Z)


# -----------------------------
# CNN with ADMM
# -----------------------------
class CNN_ADMM(nn.Module):
    def __init__(self, rho=1e-4):
        super(CNN_ADMM, self).__init__()
        self.conv = nn.Sequential(
            ADMMConv2d(3, 64, 3, padding=1, rho=rho), nn.BatchNorm2d(64), nn.ReLU(),
            ADMMConv2d(64, 64, 3, padding=1, rho=rho), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 16x16

            ADMMConv2d(64, 128, 3, padding=1, rho=rho), nn.BatchNorm2d(128), nn.ReLU(),
            ADMMConv2d(128, 128, 3, padding=1, rho=rho), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 8x8

            ADMMConv2d(128, 256, 3, padding=1, rho=rho), nn.BatchNorm2d(256), nn.ReLU(),
            ADMMConv2d(256, 256, 3, padding=1, rho=rho), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 4x4
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            ADMMLinear(256 * 4 * 4, 512, rho=rho), nn.ReLU(), nn.Dropout(0.4),
            ADMMLinear(512, 10, rho=rho)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

    def admm_loss(self):
        penalty = 0
        for module in self.modules():
            if isinstance(module, (ADMMConv2d, ADMMLinear)):
                penalty += module.admm_loss()
        return penalty

    def update_ZU(self):
        for module in self.modules():
            if isinstance(module, (ADMMConv2d, ADMMLinear)):
                module.update_ZU()


# -----------------------------
# Setup
# -----------------------------
model = CNN_ADMM(rho=1e-4).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=70)

'''# -----------------------------
# Warm-up Training
# -----------------------------
print("Warm-up phase...")
for epoch in range(10):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Warm-up Epoch [{epoch+1}/10] Loss: {total_loss/len(train_loader):.4f}")'''

# -----------------------------
# ADMM Pruning Phase
# -----------------------------
print("ADMM pruning phase...")
for epoch in range(100):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels) + model.admm_loss()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Update Z and U
    model.update_ZU()
    scheduler.step()
    print(f"ADMM Epoch [{epoch+1}/100] Loss: {total_loss/len(train_loader):.4f}")

'''# -----------------------------
# Fine-tuning
# -----------------------------
print("Fine-tuning pruned model...")
for epoch in range(10):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Fine-tune Epoch [{epoch+1}/10] Loss: {total_loss/len(train_loader):.4f}")'''

# -----------------------------
# Evaluation
# -----------------------------
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

acc = 100 * correct / total
print(f"Final Accuracy after ADMM pruning + fine-tuning: {acc:.2f}%")

# Save model
torch.save(model.state_dict(), 'cnn_admm_pruned_improved.pth')


Running on: cuda
ADMM pruning phase...
ADMM Epoch [1/100] Loss: 1.8935
ADMM Epoch [2/100] Loss: 1.5106
ADMM Epoch [3/100] Loss: 1.3533
ADMM Epoch [4/100] Loss: 1.2531
ADMM Epoch [5/100] Loss: 1.1863
ADMM Epoch [6/100] Loss: 1.1217
ADMM Epoch [7/100] Loss: 1.0796
ADMM Epoch [8/100] Loss: 1.0417
ADMM Epoch [9/100] Loss: 1.0114
ADMM Epoch [10/100] Loss: 0.9845
ADMM Epoch [11/100] Loss: 0.9607
ADMM Epoch [12/100] Loss: 0.9409
ADMM Epoch [13/100] Loss: 0.9257
ADMM Epoch [14/100] Loss: 0.9011
ADMM Epoch [15/100] Loss: 0.8899
ADMM Epoch [16/100] Loss: 0.8745
ADMM Epoch [17/100] Loss: 0.8570
ADMM Epoch [18/100] Loss: 0.8435
ADMM Epoch [19/100] Loss: 0.8339
ADMM Epoch [20/100] Loss: 0.8238
ADMM Epoch [21/100] Loss: 0.8074
ADMM Epoch [22/100] Loss: 0.8006
ADMM Epoch [23/100] Loss: 0.7891
ADMM Epoch [24/100] Loss: 0.7751
ADMM Epoch [25/100] Loss: 0.7706
ADMM Epoch [26/100] Loss: 0.7602
ADMM Epoch [27/100] Loss: 0.7491
ADMM Epoch [28/100] Loss: 0.7411
ADMM Epoch [29/100] Loss: 0.7360
ADMM Epoch [3